# 🦷 Dental AI — Full Combined System
**Tab 1:** Panoramic X-ray Analysis (YOLO → Diagnostic Agent → PDF Report)  
**Tab 2:** Single Tooth Fix (Click on tooth → SAM2 → Stable Diffusion)

In [ ]:
# Cell 1 — Install all dependencies
!pip install -q ultralytics reportlab google-generativeai gradio \
             diffusers transformers accelerate \
             opencv-python-headless Pillow sam2 ipympl
print('✅ All packages installed!')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 kB 3.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 152.8/152.8 kB 15.5 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 4.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 70.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 91.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 519.0/519.0 kB 44.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.7/154.7 kB 16.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 129.7 MB/s eta 0:00:00
✅ All packages installed!


In [ ]:
# Cell 2 — Mount Drive
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted.')

Mounted at /content/drive
Drive mounted.


In [ ]:
!pip install git+https://github.com/facebookresearch/sam2.git

  Cloning https://github.com/facebookresearch/sam2.git to /tmp/pip-req-build-zl4dyiug
  Running command git clone --filter=blob:none --quiet https://github.com/facebookresearch/sam2.git /tmp/pip-req-build-zl4dyiug
  Resolved https://github.com/facebookresearch/sam2.git to commit 2b90b9f5ceec907a1c18123530e92e794ad901a4
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for SAM-2: filename=sam_2-1.0-cp312-cp312-linux_x86_64.whl size=183669 sha256=09c239c72f97bd2687012677ef17793237cfb0851155d63f008c505d3b9742c8
  Stored in directory: /tmp/pip-ephem-wheel-cache-ddn0v986/wheels/25/a3/8a/abd69dc6a6926b5e75c24810afac36c7b49b5c0f8a100147d6
Successfully built SAM-2


In [ ]:
# Cell 3 — Imports
!pip install ultralytics
import ultralytics

import os, re, json, gc, base64, cv2, tempfile, shutil
import numpy as np
import torch
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
from io import BytesIO
from datetime import datetime
import google.generativeai as genai
from ultralytics import YOLO
from sam2.sam2_image_predictor import SAM2ImagePredictor
from diffusers import StableDiffusionInpaintPipeline
from reportlab.lib.pagesizes import A4
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.units import cm
from reportlab.lib import colors
from reportlab.platypus import (
    SimpleDocTemplate, Paragraph, Spacer,
    Table, TableStyle, HRFlowable
)
import gradio as gr

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'✅ Using: {device}')

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


✅ Using: cuda


In [ ]:
!ls /content/drive/MyDrive/DentalAI/DentalAI/

dental_yolo  weights


In [ ]:
# Cell 4 — Restore Dataset + YAML
if not os.path.exists('/content/dental_yolo/images/train'):
    print('Copying dataset from Drive...')
    if os.path.exists('/content/dental_yolo'):
        shutil.rmtree('/content/dental_yolo')
    shutil.copytree('/content/drive/MyDrive/DentalAI/DentalAI/dental_yolo', '/content/dental_yolo')
    print('Done.')
else:
    print('Dataset already in /content/dental_yolo')

yaml_content = """path: /content/dental_yolo
train: images/train
val: images/val
nc: 4
names:
  0: Impacted
  1: Caries
  2: Periapical_Lesion
  3: Deep_Caries
"""
with open('/content/dental_yolo/dental.yaml', 'w') as f:
    f.write(yaml_content)

print(f"Train: {len(os.listdir('/content/dental_yolo/images/train'))} images")
print(f"Val  : {len(os.listdir('/content/dental_yolo/images/val'))} images")
print('YAML ready.')

Copying dataset from Drive...
Done.
Train: 521 images
Val  : 2 images
YAML ready.


In [ ]:
# Cell 5 — Load all models

# ── YOLO ──────────────────────────────────────────────────────────
BEST_PT = '/content/drive/MyDrive/DentalAI/DentalAI/weights/best.pt'
yolo_model = YOLO(BEST_PT)
print(f'✅ YOLO loaded from: {BEST_PT}')

# ── Gemini ────────────────────────────────────────────────────────
GEMINI_API_KEY = 'YOUR_GEMINI_API_KEY_HERE'   # <-- paste your key
genai.configure(api_key=GEMINI_API_KEY)
gemini_model = genai.GenerativeModel('gemini-2.5-flash')
print('✅ Gemini loaded!')

# ── SAM2 ──────────────────────────────────────────────────────────
sam_predictor = SAM2ImagePredictor.from_pretrained('facebook/sam2-hiera-large')
sam_predictor.model.to(device)
print('✅ SAM2 loaded!')

# ── SD2 Inpainting ────────────────────────────────────────────────
sd_pipe = StableDiffusionInpaintPipeline.from_pretrained(
    'sd2-community/stable-diffusion-2-inpainting',
    torch_dtype=torch.float16,
).to('cuda')
sd_pipe.enable_attention_slicing()
print('✅ SD2 loaded!')

print('\n✅ ALL MODELS READY!')

✅ YOLO loaded from: /content/drive/MyDrive/DentalAI/DentalAI/weights/best.pt
✅ Gemini loaded!


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


sam2_hiera_large.pt:   0%|          | 0.00/898M [00:00<?, ?B/s]

✅ SAM2 loaded!


model_index.json:   0%|          | 0.00/544 [00:00<?, ?B/s]

Fetching 13 files:   0%|          | 0/13 [00:00<?, ?it/s]

Loading pipeline components...:   0%|          | 0/6 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/372 [00:00<?, ?it/s]

✅ SD2 loaded!

✅ ALL MODELS READY!


In [ ]:
# Cell 6 — YOLO + FDI helpers

DISEASE_MAP = {
    0: 'Impacted',
    1: 'Caries',
    2: 'Periapical Lesion',
    3: 'Deep Caries'
}

DISEASE_COLORS_HEX = {
    'Impacted':          '#FFD700',
    'Caries':            '#FF4444',
    'Periapical Lesion': '#FF8800',
    'Deep Caries':       '#FF44FF'
}

URGENCY_COLOR = {
    'URGENT':  '#ff4444',
    'SOON':    '#ff8800',
    'ROUTINE': '#44aa44'
}


def get_fdi_from_position(x_center, y_center, img_width, img_height):
    norm_x = x_center / img_width
    norm_y = y_center / img_height
    if norm_y < 0.5:
        if norm_x < 0.5:
            quadrant = 1
            tooth = max(1, min(8, int((0.5 - norm_x) / 0.5 * 8) + 1))
        else:
            quadrant = 2
            tooth = max(1, min(8, int((norm_x - 0.5) / 0.5 * 8) + 1))
    else:
        if norm_x < 0.5:
            quadrant = 4
            tooth = max(1, min(8, int((0.5 - norm_x) / 0.5 * 8) + 1))
        else:
            quadrant = 3
            tooth = max(1, min(8, int((norm_x - 0.5) / 0.5 * 8) + 1))
    return quadrant * 10 + tooth


def analyze_xray(image_path, conf_threshold=0.25):
    img = Image.open(image_path).convert('RGB')
    img_array = np.array(img)
    img_h, img_w = img_array.shape[:2]
    results = yolo_model(image_path, conf=conf_threshold, iou=0.45, verbose=False)[0]
    detections = []
    for box in results.boxes:
        x1, y1, x2, y2 = box.xyxy[0].tolist()
        cx = (x1 + x2) / 2
        cy = (y1 + y2) / 2
        class_id   = int(box.cls[0])
        confidence = float(box.conf[0])
        disease    = DISEASE_MAP[class_id]
        fdi        = get_fdi_from_position(cx, cy, img_w, img_h)
        detections.append({
            'fdi':        fdi,
            'disease':    disease,
            'confidence': round(confidence, 3),
            'bbox':       [x1, y1, x2, y2]
        })
    detections.sort(key=lambda x: x['fdi'])
    return detections, img_array


print('✅ YOLO helpers ready!')

✅ YOLO helpers ready!


In [ ]:
# Cell 7 — Diagnostic Agent (Gemini)

SYSTEM_PROMPT = """You are a senior dental radiologist and clinical decision support AI.
You receive structured detection results from a YOLOv11 model that analyzed a panoramic dental X-ray (OPG).

Your job is to:
1. Assess overall dental health status
2. Prioritize findings by clinical urgency
3. Generate a phased treatment plan
4. Assess orthodontic readiness

Disease context:
- Deep Caries: highest urgency → RCT + Crown
- Periapical Lesion: active infection → RCT + Antibiotics
- Caries: moderate → Composite restoration
- Impacted: surgical assessment → Extraction or monitoring

Respond ONLY with valid JSON, no markdown, no extra text:
{
  "health_status": "string",
  "urgency": "URGENT | SOON | ROUTINE",
  "total_findings": integer,
  "summary": {
    "caries_count": integer,
    "deep_caries_count": integer,
    "impacted_count": integer,
    "infection_count": integer
  },
  "clinical_notes": "string",
  "treatment_plan": [
    {"phase": integer, "title": "string", "items": ["string"]}
  ],
  "orthodontic_assessment": "string"
}"""


def run_diagnostic_agent(detections):
    findings_text = '\n'.join([
        f"- Tooth {d['fdi']}: {d['disease']} (confidence {d['confidence']:.1%})"
        for d in detections
    ])
    prompt = f"{SYSTEM_PROMPT}\n\nPanoramic X-ray findings:\n{findings_text}\n\nTotal detections: {len(detections)}"
    response = gemini_model.generate_content(prompt)
    raw = response.text.strip()
    if raw.startswith('```'):
        raw = raw.split('\n', 1)[1]
        raw = raw.rsplit('```', 1)[0]
    return json.loads(raw)


print('✅ Diagnostic agent ready!')

✅ Diagnostic agent ready!


In [ ]:
# Cell 8 — PDF Report Generator

def generate_pdf_report(detections, diagnosis, output_path, patient_name='Patient'):
    doc    = SimpleDocTemplate(output_path, pagesize=A4,
                               rightMargin=2*cm, leftMargin=2*cm,
                               topMargin=2*cm,   bottomMargin=2*cm)
    styles = getSampleStyleSheet()
    story  = []

    title_style  = ParagraphStyle('Title2',  parent=styles['Title'],
                                  fontSize=18, textColor=colors.HexColor('#1a3c5e'), spaceAfter=8)
    header_style = ParagraphStyle('Header2', parent=styles['Heading2'],
                                  fontSize=12, textColor=colors.HexColor('#1a3c5e'),
                                  spaceBefore=12, spaceAfter=6)
    normal_style = ParagraphStyle('Normal2', parent=styles['Normal'], fontSize=10, leading=15)
    small_style  = ParagraphStyle('Small',   parent=styles['Normal'], fontSize=8, textColor=colors.grey)

    story.append(Paragraph('Dental AI — Diagnostic Report', title_style))
    story.append(Paragraph(f'Patient : {patient_name}', normal_style))
    story.append(Paragraph(f'Date    : {datetime.now().strftime("%B %d, %Y")}', normal_style))
    story.append(Paragraph('System  : YOLOv11x + Gemini Diagnostic Agent', normal_style))
    story.append(HRFlowable(width='100%', thickness=1, color=colors.HexColor('#1a3c5e')))
    story.append(Spacer(1, 0.4*cm))

    story.append(Paragraph('SECTION A — Clinical Report (For Dentist)', header_style))
    status_data = [
        ['Overall Health Status', diagnosis['health_status']],
        ['Urgency Level',         diagnosis['urgency']],
        ['Total Findings',        str(diagnosis['total_findings'])],
        ['Caries',                str(diagnosis['summary']['caries_count'])],
        ['Deep Caries',           str(diagnosis['summary']['deep_caries_count'])],
        ['Impacted Teeth',        str(diagnosis['summary']['impacted_count'])],
        ['Active Infections',     str(diagnosis['summary']['infection_count'])],
    ]
    st = Table(status_data, colWidths=[6*cm, 10*cm])
    st.setStyle(TableStyle([
        ('BACKGROUND', (0, 0), (0, -1), colors.HexColor('#f0f4f8')),
        ('FONTNAME',   (0, 0), (0, -1), 'Helvetica-Bold'),
        ('FONTSIZE',   (0, 0), (-1, -1), 10),
        ('GRID',       (0, 0), (-1, -1), 0.5, colors.grey),
        ('PADDING',    (0, 0), (-1, -1), 6),
    ]))
    story.append(st)
    story.append(Spacer(1, 0.4*cm))

    story.append(Paragraph('Clinical Notes:', header_style))
    story.append(Paragraph(diagnosis['clinical_notes'], normal_style))
    story.append(Spacer(1, 0.3*cm))

    story.append(Paragraph('Detailed Findings:', header_style))
    treatment_lookup = {
        'Deep Caries':       'RCT + Crown',
        'Periapical Lesion': 'RCT + Antibiotics',
        'Caries':            'Composite restoration',
        'Impacted':          'Extraction / Monitor'
    }
    findings_data = [['Tooth (FDI)', 'Diagnosis', 'Confidence', 'Treatment']]
    for det in detections:
        findings_data.append([
            f"Tooth {det['fdi']}", det['disease'],
            f"{det['confidence']:.1%}",
            treatment_lookup.get(det['disease'], 'Consult specialist')
        ])
    ft = Table(findings_data, colWidths=[3.5*cm, 4.5*cm, 3*cm, 5.5*cm])
    ft.setStyle(TableStyle([
        ('BACKGROUND',     (0, 0), (-1, 0),  colors.HexColor('#1a3c5e')),
        ('TEXTCOLOR',      (0, 0), (-1, 0),  colors.white),
        ('FONTNAME',       (0, 0), (-1, 0),  'Helvetica-Bold'),
        ('FONTSIZE',       (0, 0), (-1, -1), 9),
        ('GRID',           (0, 0), (-1, -1), 0.5, colors.grey),
        ('PADDING',        (0, 0), (-1, -1), 5),
        ('ROWBACKGROUNDS', (0, 1), (-1, -1), [colors.white, colors.HexColor('#f9f9f9')]),
    ]))
    story.append(ft)
    story.append(Spacer(1, 0.4*cm))

    story.append(Paragraph('Treatment Plan:', header_style))
    for phase in diagnosis['treatment_plan']:
        story.append(Paragraph(f"<b>{phase['title']}:</b>", normal_style))
        for item in phase['items']:
            story.append(Paragraph(f'• {item}', normal_style))
        story.append(Spacer(1, 0.2*cm))

    story.append(Paragraph('Orthodontic Assessment:', header_style))
    story.append(Paragraph(diagnosis['orthodontic_assessment'], normal_style))
    story.append(HRFlowable(width='100%', thickness=0.5, color=colors.grey))
    story.append(Spacer(1, 0.4*cm))

    story.append(Paragraph('SECTION B — Patient Summary', header_style))
    story.append(Paragraph(
        f'Dear {patient_name}, here is a simple explanation of your dental X-ray results:',
        normal_style
    ))
    story.append(Spacer(1, 0.2*cm))
    patient_explanations = {
        'Caries':            'has a cavity that needs to be filled',
        'Deep Caries':       'has a deep cavity that may need a root canal treatment',
        'Impacted':          'is stuck under the gum and may need to be removed',
        'Periapical Lesion': 'has an infection at the root that needs urgent treatment'
    }
    story.append(Paragraph('Teeth that need attention:', normal_style))
    for det in detections:
        exp = patient_explanations.get(det['disease'], 'needs dental attention')
        story.append(Paragraph(f"• <b>Tooth {det['fdi']}</b> {exp}.", normal_style))
    story.append(Spacer(1, 0.4*cm))
    story.append(Paragraph(
        'This report was generated by an AI system and must be reviewed by a qualified dentist '
        'before any clinical decision is made.',
        small_style
    ))
    doc.build(story)
    print(f'PDF saved to: {output_path}')


print('✅ PDF generator ready!')

✅ PDF generator ready!


In [ ]:
# # Cell 9 — Single Tooth Fix helpers (SAM2 + SD2)
# # Cell 9 — Single Tooth Fix helpers (SAM2 + SD2)

# TOOTH_PROMPTS = {
#     'cavity'  : 'a single natural human tooth, healthy white enamel, no cavity, no decay, smooth surface, realistic dental photo, clinic lighting, macro photography, 4k',
#     'yellow'  : 'a single natural human tooth, bright white enamel, clean healthy tooth, professional whitening result, realistic dental photo, clinic lighting, macro photography, 4k',
#     'broken'  : 'a single perfect human tooth, complete intact tooth, no chips, no cracks, no missing pieces, fully restored smooth tooth, uniform white enamel surface, natural dental photo, clinic lighting, macro photography, 4k sharp',
#     'crooked' : 'a single natural human tooth, perfectly straight aligned tooth, correct position, realistic dental photo, clinic lighting, macro photography, 4k',
#     'gap'     : 'a single natural human tooth, aligned tooth, closed gap, uniform spacing, realistic dental photo, clinic lighting, macro photography, 4k',
#     'missing' : 'a single natural human tooth, complete natural white tooth, healthy root, realistic dental photo, clinic lighting, macro photography, 4k',
# }

# TOOTH_NEGATIVE = (
#     'cavity, decay, yellow, stained, broken, crack, crooked, gap, missing tooth, '
#     'cartoon, illustration, drawing, blurry, fake, plastic, deformed, braces, brackets, '
#     'watermark, text, extra teeth, unrealistic, CGI, 3d render, painting, sketch, '
#     'red, blood, gum, dark background, shadow, blur'
#     'broken, chipped, cracked, missing piece, cavity, decay, yellow, stained, '
#     'cartoon, illustration, drawing, blurry, fake, plastic, deformed, braces, brackets, '
#     'watermark, text, extra teeth, unrealistic, CGI, 3d render, painting, sketch, '
#     'red, blood, dark background, shadow, blur'
# )

# tooth_original_state = {'image': None}


# def blend_result(original_pil, generated_pil, mask_np):
#     orig    = np.array(original_pil.resize(generated_pil.size)).astype(np.float32)
#     gen     = np.array(generated_pil).astype(np.float32)
#     m       = cv2.resize(mask_np, (generated_pil.width, generated_pil.height)).astype(np.float32) / 255.0
#     m_u8    = (m * 255).astype(np.uint8)
#     m_blur  = cv2.GaussianBlur(m_u8, (31, 31), 0)
#     m_erode = cv2.erode(m_blur, np.ones((3, 3), np.uint8), iterations=1)
#     m_final = (m_erode / 255.0)[:, :, np.newaxis]
#     blended = (gen * m_final + orig * (1 - m_final)).astype(np.uint8)
#     return Image.fromarray(blended)


# def segment_tooth_at_click(image_np, click_x, click_y):
#     h, w = image_np.shape[:2]
#     sam_predictor.set_image(image_np)
#     with torch.inference_mode():
#         masks, scores, _ = sam_predictor.predict(
#             point_coords     = np.array([[click_x, click_y]]),
#             point_labels     = np.array([1]),
#             multimask_output = True,
#         )
#     for idx in np.argsort(scores)[::-1]:
#         candidate = (masks[idx] * 255).astype(np.uint8)
#         coverage  = np.sum(candidate > 127) / (w * h) * 100
#         if coverage <= 15:
#             kernel    = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7, 7))
#             candidate = cv2.morphologyEx(candidate, cv2.MORPH_CLOSE, kernel)
#             candidate = cv2.morphologyEx(candidate, cv2.MORPH_OPEN,  kernel)
#             return candidate, coverage
#     return None, 0.0


# def fix_single_tooth(image_np, click_x, click_y, problem):
#     img_pil = Image.fromarray(image_np)

#     tooth_mask, coverage = segment_tooth_at_click(image_np, click_x, click_y)
#     if tooth_mask is None:
#         return None, 'SAM2: mask too large — click more precisely on the tooth center.'

#     prompt   = TOOTH_PROMPTS.get(problem, TOOTH_PROMPTS['cavity'])
#     mask_pil = Image.fromarray(tooth_mask)
#     img_512  = img_pil.resize((512, 512))
#     mask_512 = mask_pil.resize((512, 512))

#     results = []
#     for i in range(4):
#         gen = torch.Generator('cuda').manual_seed(42 + i * 7)
#         out = sd_pipe(
#             prompt                = prompt,
#             negative_prompt       = TOOTH_NEGATIVE,
#             image                 = img_512,
#             mask_image            = mask_512,
#             guidance_scale        = 8.5,
#             num_inference_steps   = 50,
#             strength              = 0.80,
#             padding_mask_crop     = 16,
#             num_images_per_prompt = 1,
#             generator             = gen,
#         ).images[0]
#         results.append(blend_result(img_pil, out, tooth_mask))

#     fig, axes = plt.subplots(1, 5, figsize=(24, 5))
#     axes[0].imshow(img_pil); axes[0].set_title('Original', fontweight='bold'); axes[0].axis('off')
#     axes[0].plot(click_x, click_y, 'g*', markersize=15)
#     for i, img in enumerate(results):
#         axes[i+1].imshow(img); axes[i+1].set_title(f'Result {i+1}'); axes[i+1].axis('off')
#     plt.suptitle(f'Single Tooth Fix — {problem}', fontsize=13, fontweight='bold')
#     plt.tight_layout()
#     grid_path = '/content/tooth_fix_results.png'
#     plt.savefig(grid_path, dpi=120, bbox_inches='tight')
#     plt.close()

#     for i, img in enumerate(results):
#         img.save(f'/content/tooth_fixed_{problem}_{i+1}.png')

#     return grid_path, f'✅ Done! Coverage: {coverage:.1f}%. Saved tooth_fixed_{problem}_1-4.png'


# print('✅ Single tooth fix helpers ready!')






















# Cell 9 — Single Tooth Analysis (SAM2 + Gemini)

import base64
from io import BytesIO

tooth_original_state = {'image': None}
tooth_click_state    = {'x': None, 'y': None}


def pil_to_base64(pil_img, fmt='PNG'):
    buffer = BytesIO()
    pil_img.save(buffer, format=fmt)
    return base64.b64encode(buffer.getvalue()).decode('utf-8')


def segment_tooth_at_click(image_np, click_x, click_y):
    h, w = image_np.shape[:2]
    sam_predictor.set_image(image_np)
    with torch.inference_mode():
        masks, scores, _ = sam_predictor.predict(
            point_coords     = np.array([[click_x, click_y]]),
            point_labels     = np.array([1]),
            multimask_output = True,
        )
    for idx in np.argsort(scores)[::-1]:
        candidate = (masks[idx] * 255).astype(np.uint8)
        coverage  = np.sum(candidate > 127) / (w * h) * 100
        if coverage <= 25:
            kernel    = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7, 7))
            candidate = cv2.morphologyEx(candidate, cv2.MORPH_CLOSE, kernel)
            candidate = cv2.morphologyEx(candidate, cv2.MORPH_OPEN,  kernel)
            return candidate, coverage
    return None, 0.0


def analyze_single_tooth(image_np, click_x, click_y):
    img_pil = Image.fromarray(image_np)

    # SAM2 — segment the tooth
    tooth_mask, coverage = segment_tooth_at_click(image_np, click_x, click_y)
    if tooth_mask is None:
        return None, None, 'SAM2: could not isolate a single tooth — click more precisely on the tooth center.'

    # Crop tooth region with padding
    ys, xs  = np.where(tooth_mask > 127)
    pad     = 30
    x1      = max(0, xs.min() - pad)
    x2      = min(image_np.shape[1], xs.max() + pad)
    y1      = max(0, ys.min() - pad)
    y2      = min(image_np.shape[0], ys.max() + pad)
    crop_np = image_np[y1:y2, x1:x2]
    crop_pil = Image.fromarray(crop_np)

    # Overlay mask on original for context
    overlay = image_np.copy()
    overlay[tooth_mask > 127] = (overlay[tooth_mask > 127] * 0.5 + np.array([0, 200, 100]) * 0.5).astype(np.uint8)
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    axes[0].imshow(image_np);   axes[0].set_title('Original');      axes[0].axis('off')
    axes[0].plot(click_x, click_y, 'g*', markersize=15)
    axes[1].imshow(overlay);    axes[1].set_title('Selected Tooth (SAM2)'); axes[1].axis('off')
    axes[2].imshow(crop_np);    axes[2].set_title('Tooth Crop');   axes[2].axis('off')
    plt.suptitle(f'SAM2 Segmentation — coverage: {coverage:.1f}%', fontsize=13, fontweight='bold')
    plt.tight_layout()
    seg_path = '/content/tooth_segmented.png'
    plt.savefig(seg_path, dpi=120, bbox_inches='tight')
    plt.close()

    # Gemini — analyze the cropped tooth
    full_b64 = pil_to_base64(img_pil)
    crop_b64 = pil_to_base64(crop_pil)

    analysis_prompt = """You are an expert dental clinician analyzing a patient's tooth photo.

You will receive two images:
- Image 1: the full mouth photo with the selected tooth highlighted
- Image 2: a close-up crop of that specific tooth

Analyze the tooth carefully and provide a detailed clinical assessment.

Respond in this exact format:

## Tooth Assessment

**Condition:** [describe what you see — color, shape, damage, cracks, decay, etc.]

**Diagnosis:** [your clinical diagnosis]

**Severity:** [Mild / Moderate / Severe]

**Recommended Treatment:**
- [treatment step 1]
- [treatment step 2]
- [treatment step 3 if needed]

**Expected Outcome:**
[describe what the tooth should look like after proper treatment]

**Urgency:** [Routine / Soon / Urgent]

**Clinical Notes:**
[any additional observations or warnings]

---
*This assessment is AI-generated and must be reviewed by a qualified dentist before any clinical decision.*"""

    response = gemini_model.generate_content([
        {'mime_type': 'image/png', 'data': full_b64},
        {'mime_type': 'image/png', 'data': crop_b64},
        analysis_prompt
    ])

    return seg_path, crop_pil, response.text


print('✅ Single tooth analysis ready!')

✅ Single tooth analysis ready!


In [ ]:
# Cell 10 — Gradio Tab 1: X-ray Analysis

def process_xray(image, patient_name):
    if image is None:
        return None, 'No image uploaded.', None
    if not patient_name.strip():
        patient_name = 'Patient'

    tmp = tempfile.NamedTemporaryFile(suffix='.png', delete=False)
    Image.fromarray(image).save(tmp.name)

    detections, img_array = analyze_xray(tmp.name)

    # Annotated image
    fig, ax = plt.subplots(1, 1, figsize=(16, 7))
    ax.imshow(img_array, cmap='gray')
    for det in detections:
        x1, y1, x2, y2 = det['bbox']
        color = DISEASE_COLORS_HEX[det['disease']]
        rect  = patches.Rectangle((x1, y1), x2-x1, y2-y1,
                                   linewidth=2, edgecolor=color, facecolor='none')
        ax.add_patch(rect)
        ax.text(x1, y1-5,
                f"T{det['fdi']} {det['disease']} ({det['confidence']:.0%})",
                color=color, fontsize=7, fontweight='bold',
                bbox=dict(boxstyle='round,pad=0.2', facecolor='black', alpha=0.6))
    ax.set_title(f'Dental AI — {len(detections)} findings detected', fontsize=13)
    ax.axis('off')
    plt.tight_layout()
    annotated_path = '/content/annotated.png'
    plt.savefig(annotated_path, dpi=150, bbox_inches='tight')
    plt.close()

    # Diagnostic agent
    diagnosis = run_diagnostic_agent(detections)

    # Markdown report
    urgency_color = URGENCY_COLOR.get(diagnosis['urgency'], 'grey')
    findings_md = f"""
## Diagnostic Report

**Health Status:** {diagnosis['health_status']}
**Urgency:** <span style='color:{urgency_color}'>{diagnosis['urgency']}</span>
**Total Findings:** {diagnosis['total_findings']}

---

### Summary
| Type | Count |
|------|-------|
| Caries | {diagnosis['summary']['caries_count']} |
| Deep Caries | {diagnosis['summary']['deep_caries_count']} |
| Impacted | {diagnosis['summary']['impacted_count']} |
| Active Infections | {diagnosis['summary']['infection_count']} |

---

### Detected Teeth
"""
    for det in detections:
        findings_md += f"- **Tooth {det['fdi']}** → {det['disease']} ({det['confidence']:.0%})\n"

    findings_md += f"""
---

### Clinical Notes
{diagnosis['clinical_notes']}

---

### Treatment Plan
"""
    for phase in diagnosis['treatment_plan']:
        findings_md += f"\n**{phase['title']}:**\n"
        for item in phase['items']:
            findings_md += f"- {item}\n"

    findings_md += f"""
---

### Orthodontic Assessment
{diagnosis['orthodontic_assessment']}

---
*This report is AI-generated and must be reviewed by a qualified dentist.*
"""

    # PDF
    pdf_path = '/content/dental_report.pdf'
    generate_pdf_report(detections, diagnosis, pdf_path, patient_name=patient_name)

    os.unlink(tmp.name)
    return annotated_path, findings_md, pdf_path


print('✅ Tab 1 function ready!')

✅ Tab 1 function ready!


In [ ]:
import gradio as gr
print(gr.__version__)

5.50.0


In [ ]:
# Cell 11 — Gradio Tab 2: Single Tooth Fix
#
# The doctor uploads the photo, then uses the X and Y sliders to
# position the crosshair on the tooth they want to fix.
# Gradio's ImageEditor .select event gives us the click coordinates directly.

# Store last click coordinates across Gradio calls
# Cell 11 — Gradio Tab 2 functions

# tooth_click_state    = {'x': None, 'y': None}
# tooth_original_state = {'image': None}


# def capture_tooth_click(image, evt: gr.SelectData):
#     if image is None:
#         return image, 'Upload an image first.'
#     x, y = evt.index[0], evt.index[1]
#     tooth_click_state['x'] = x
#     tooth_click_state['y'] = y
#     tooth_original_state['image'] = image.copy()  # save clean original

#     # dot on preview only
#     preview = image.copy()
#     cv2.circle(preview, (x, y), 12, (0, 255, 0), -1)
#     cv2.circle(preview, (x, y), 16, (255, 255, 255), 2)
#     return preview, f'✅ Tooth selected at ({x}, {y}) — choose problem and click Fix Tooth.'


# def run_tooth_fix(image, problem):
#     if tooth_original_state['image'] is None:
#         return None, 'Click on the tooth in the image above first.'
#     if tooth_click_state['x'] is None:
#         return None, 'Click on the tooth first.'

#     x, y        = tooth_click_state['x'], tooth_click_state['y']
#     clean_image = tooth_original_state['image']   # no dot
#     grid_path, msg = fix_single_tooth(clean_image, x, y, problem)
#     return grid_path, msg


# print('✅ Tab 2 functions ready!')












# Cell 11 — Gradio Tab 2 functions

tooth_click_state    = {'x': None, 'y': None}
tooth_original_state = {'image': None}


def capture_tooth_click(image, evt: gr.SelectData):
    if image is None:
        return image, 'Upload an image first.'
    x, y = evt.index[0], evt.index[1]
    tooth_click_state['x'] = x
    tooth_click_state['y'] = y
    tooth_original_state['image'] = image.copy()

    preview = image.copy()
    cv2.circle(preview, (x, y), 12, (0, 255, 0), -1)
    cv2.circle(preview, (x, y), 16, (255, 255, 255), 2)
    return preview, f'✅ Tooth selected at ({x}, {y}) — click Analyze Tooth.'


def run_tooth_analysis(image, _problem):
    if tooth_original_state['image'] is None:
        return None, 'No analysis yet.', 'Click on the tooth first.'
    if tooth_click_state['x'] is None:
        return None, 'No analysis yet.', 'Click on the tooth first.'

    x, y        = tooth_click_state['x'], tooth_click_state['y']
    clean_image = tooth_original_state['image']
    seg_path, crop_pil, report = analyze_single_tooth(clean_image, x, y)

    if seg_path is None:
        return None, 'No analysis yet.', report

    return seg_path, report, '✅ Analysis complete!'


print('✅ Tab 2 functions ready!')

✅ Tab 2 functions ready!


In [ ]:
# Cell 12 — Launch Gradio App

with gr.Blocks(theme=gr.themes.Soft(), title='Dental AI') as demo:

    gr.Markdown("""
    # 🦷 Dental AI — Clinical Decision Support
    ### YOLOv11x + Gemini + SAM2 + Stable Diffusion
    """)

    # ── Tab 1: X-ray Analysis ─────────────────────────────────────
    with gr.Tab('📋 X-ray Analysis'):
        gr.Markdown('Upload a panoramic dental X-ray to get a full AI diagnostic report.')
        with gr.Row():
            with gr.Column(scale=1):
                xray_input    = gr.Image(label='Upload Panoramic X-ray', type='numpy')
                patient_name  = gr.Textbox(label='Patient Name', placeholder='e.g. Ahmed Mohamed', value='Patient')
                analyze_btn   = gr.Button('Analyze X-ray', variant='primary', size='lg')
            with gr.Column(scale=2):
                xray_output   = gr.Image(label='Annotated X-ray')
        with gr.Row():
            report_output = gr.Markdown(label='Diagnostic Report')
        with gr.Row():
            pdf_output    = gr.File(label='Download PDF Report')

        analyze_btn.click(
            fn      = process_xray,
            inputs  = [xray_input, patient_name],
            outputs = [xray_output, report_output, pdf_output]
        )

    # ── Tab 2: Single Tooth Fix ───────────────────────────────────
    # ── Tab 2: Single Tooth Analysis ─────────────────────────────
        with gr.Tab('🦷 Single Tooth Analysis'):
            gr.Markdown("""
            **How to use:**
            1. Upload the patient photo
            2. Click directly on the tooth you want to analyze
            3. Click Analyze Tooth
            """)
            with gr.Row():
                with gr.Column(scale=1):
                    tooth_input  = gr.Image(label='Upload Patient Photo — Click on a tooth', type='numpy')
                    click_status = gr.Textbox(label='Click Status', interactive=False, value='Click on a tooth above.')
                    analyze_tooth_btn = gr.Button('Analyze Tooth', variant='primary', size='lg')
                    fix_status   = gr.Textbox(label='Status', interactive=False)
                with gr.Column(scale=2):
                    tooth_seg_output = gr.Image(label='SAM2 Segmentation')
                    tooth_report     = gr.Markdown(label='Gemini Clinical Report')

            tooth_input.select(
                fn      = capture_tooth_click,
                inputs  = [tooth_input],
                outputs = [tooth_input, click_status]
            )

            analyze_tooth_btn.click(
                fn      = run_tooth_analysis,
                inputs  = [tooth_input, gr.State(None)],
                outputs = [tooth_seg_output, tooth_report, fix_status]
            )

demo.launch(share=True, debug=True)

/tmp/ipykernel_985/1765308933.py:3: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft(), title='Dental AI') as demo:


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://e1ae23533a1a7f053f.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
